# RAG System Over Your Own Documents

This notebook builds a full Retrieval-Augmented Generation (RAG) pipeline over
the files in `my_docs/`, reusing the same building blocks as Exercise 1:

- **Embeddings**: `SentenceTransformer('all-MiniLM-L6-v2')`
- **Generation**: a HuggingFace `transformers` `pipeline`
- **Orchestration**: LangChain (LCEL)

**Pipeline stages**

1. Load documents (`.md` / `.txt`) from `my_docs/`
2. Chunk documents into overlapping passages
3. Embed chunks and build a retriever
4. Generate an answer from retrieved context
5. Wrap everything in a LangChain chain
6. Evaluate retrieval + answer quality on a small Q&A test set


## 0. Setup

Install dependencies (skip if already installed in your environment).

In [1]:
!pip install -q sentence-transformers transformers langchain langchain-community \
    faiss-cpu accelerate


In [2]:
import os
import glob
import textwrap
from dataclasses import dataclass

import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from transformers import pipeline as hf_pipeline

from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline

DOCS_DIR = "my_docs"


c:\Users\rahul\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\rahul\AppData\Local\Temp\ipykernel_17512\2765557384.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 1. Load documents

We load every `.md` and `.txt` file in `my_docs/` into LangChain `Document`
objects, keeping the source filename as metadata so we can trace answers
back to their origin later.

In [4]:
def resolve_docs_dir(docs_dir: str) -> str:
    """Find my_docs/ even if the notebook's cwd isn't its parent folder.

    Checks, in order: the given path as-is, then the same path relative to
    the directory this notebook lives in (Jupyter sets cwd to the notebook's
    folder by default, but some setups - e.g. running from a different
    working directory in VS Code - do not).
    """
    candidates = [docs_dir, os.path.join(os.getcwd(), docs_dir)]
    try:
        notebook_dir = os.path.abspath("")
        candidates.append(os.path.join(notebook_dir, docs_dir))
    except Exception:
        pass

    for c in candidates:
        if os.path.isdir(c) and (glob.glob(os.path.join(c, "*.md")) or
                                  glob.glob(os.path.join(c, "*.txt"))):
            return c

    raise FileNotFoundError(
        f"Could not find a non-empty '{docs_dir}' folder.\n"
        f"Current working directory: {os.getcwd()}\n"
        f"Tried: {candidates}\n"
        f"Make sure the 'my_docs' folder (with the .md/.txt files) sits "
        f"next to this notebook, or update DOCS_DIR to an absolute path."
    )


def load_documents(docs_dir: str) -> list[Document]:
    docs_dir = resolve_docs_dir(docs_dir)
    paths = sorted(glob.glob(os.path.join(docs_dir, "*.md")) +
                    glob.glob(os.path.join(docs_dir, "*.txt")))
    docs = []
    for path in paths:
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        docs.append(Document(page_content=text, metadata={"source": os.path.basename(path)}))
    return docs

raw_docs = load_documents(DOCS_DIR)
assert len(raw_docs) > 0, "No documents loaded - check DOCS_DIR and the my_docs/ folder location."

print(f"Loaded {len(raw_docs)} documents:")
for d in raw_docs:
    print(f"  - {d.metadata['source']}  ({len(d.page_content)} chars)")


Loaded 3 documents:
  - company_handbook.md  (2350 chars)
  - onboarding_guide.txt  (2625 chars)
  - product_faq.md  (2647 chars)


## 2. Chunk documents

We split each document into overlapping chunks using
`RecursiveCharacterTextSplitter`. Overlap helps preserve context that would
otherwise be cut across a chunk boundary (e.g. a sentence split mid-way).

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(raw_docs)
print(f"Split {len(raw_docs)} documents into {len(chunks)} chunks.\n")

for c in chunks[:3]:
    print(f"--- source: {c.metadata['source']} ---")
    print(textwrap.shorten(c.page_content, width=200))
    print()


Split 3 documents into 24 chunks.

--- source: company_handbook.md ---
# Acme Robotics — Employee Handbook ## Remote Work Policy Acme Robotics operates on a hybrid work model. Employees are expected to be in the office on Tuesdays, Wednesdays, and Thursdays. Monday [...]

--- source: company_handbook.md ---
Employees who wish to work fully remote must submit a Remote Work Request form to their manager and to HR at least two weeks in advance. Approval is granted on a case-by-case basis depending on [...]

--- source: company_handbook.md ---
## Vacation and Paid Time Off Full-time employees accrue 18 days of paid vacation per year, plus 10 company holidays. Vacation accrues at a rate of 1.5 days per month and can be carried over up [...]



## 3. Embed & retrieve

We wrap `SentenceTransformer('all-MiniLM-L6-v2')` in a small adapter class
that implements LangChain's `Embeddings` interface (`embed_documents` /
`embed_query`), then build a FAISS vector store over the chunks. The
vector store is exposed as a LangChain `retriever`.

In [6]:
class SentenceTransformerEmbeddings(Embeddings):
    """Adapter so SentenceTransformer plugs into LangChain's Embeddings interface."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        embeddings = self.model.encode(texts, convert_to_numpy=True, show_progress_bar=False)
        return embeddings.tolist()

    def embed_query(self, text: str) -> list[float]:
        return self.model.encode([text], convert_to_numpy=True)[0].tolist()


embeddings = SentenceTransformerEmbeddings("all-MiniLM-L6-v2")

assert len(chunks) > 0, "No chunks to embed - 'chunks' is empty. Re-run the load/chunk cells above."

vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Vector store built with", vectorstore.index.ntotal, "vectors.")


Vector store built with 24 vectors.


In [7]:
# Quick sanity check: does retrieval pull back relevant chunks?
test_query = "How many vacation days do employees get?"
results = retriever.invoke(test_query)

for r in results:
    print(f"[{r.metadata['source']}]")
    print(textwrap.shorten(r.page_content, width=180))
    print()


[company_handbook.md]
## Vacation and Paid Time Off Full-time employees accrue 18 days of paid vacation per year, plus 10 company holidays. Vacation accrues at a rate of 1.5 days per month and can [...]

[company_handbook.md]
Sick leave is separate from vacation and is uncapped for the first 5 days of any illness; after that, employees should coordinate with HR regarding short-term disability [...]

[company_handbook.md]
# Acme Robotics — Employee Handbook ## Remote Work Policy Acme Robotics operates on a hybrid work model. Employees are expected to be in the office on Tuesdays, Wednesdays, [...]



## 4. Generate an answer from retrieved context

We use a small HuggingFace text2text model (`google/flan-t5-base`) through
`transformers.pipeline`, then wrap it with `HuggingFacePipeline` so it
behaves as a LangChain LLM. Swap in a different model name if you have
access to something larger (e.g. an instruction-tuned causal LM).

In [8]:
generator = hf_pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200,
)

llm = HuggingFacePipeline(pipeline=generator)


Device set to use cpu
C:\Users\rahul\AppData\Local\Temp\ipykernel_17512\2921972028.py:7: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=generator)


In [9]:
PROMPT_TEMPLATE = """You are a helpful assistant answering questions using ONLY the
context provided below. If the answer is not contained in the context, say
"I don't know based on the provided documents."

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate.from_template(PROMPT_TEMPLATE)


def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d in docs)


## 5. Wrap it all in a LangChain chain (LCEL)

This composes retrieval, prompt formatting, generation, and output parsing
into a single runnable chain: `rag_chain.invoke(question)`.

In [10]:
rag_chain = (
    {"context": retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [11]:
def ask(question: str, show_sources: bool = True) -> str:
    answer = rag_chain.invoke(question)
    print(f"Q: {question}\nA: {answer}\n")
    if show_sources:
        srcs = retriever.invoke(question)
        print("Sources used:")
        for s in srcs:
            print(f"  - {s.metadata['source']}")
    return answer


_ = ask("How many vacation days do employees get per year?")


Q: How many vacation days do employees get per year?
A: 18 days

Sources used:
  - company_handbook.md
  - company_handbook.md
  - company_handbook.md


In [12]:
_ = ask("What encryption does CloudSync Pro use to protect files at rest?")


Q: What encryption does CloudSync Pro use to protect files at rest?
A: AES-256.

Sources used:
  - product_faq.md
  - product_faq.md
  - product_faq.md


In [13]:
_ = ask("What training must a new employee complete in their first 3 days?")


Q: What training must a new employee complete in their first 3 days?
A: 1. Information Security Basics (45 minutes) 2. Code of Conduct and Anti-Harassment Training (60 minutes) 3. Data Privacy Fundamentals (30 minutes)

Sources used:
  - onboarding_guide.txt
  - onboarding_guide.txt
  - onboarding_guide.txt


## 6. Evaluate

We evaluate two things on a small hand-labeled test set:

1. **Retrieval quality** — did the retriever pull back a chunk from the
   correct source document?
2. **Answer quality** — does the generated answer contain the expected
   key fact(s)? (A simple keyword-containment check; swap in ROUGE/BLEU or
   an LLM-judge for a more rigorous evaluation.)


In [14]:
eval_set = [
    {
        "question": "How many vacation days do full-time employees accrue per year?",
        "expected_source": "company_handbook.md",
        "expected_keywords": ["18"],
    },
    {
        "question": "What is the price of the CloudSync Pro Business plan?",
        "expected_source": "product_faq.md",
        "expected_keywords": ["15"],
    },
    {
        "question": "How long do new employees have to enroll in benefits?",
        "expected_source": "onboarding_guide.txt",
        "expected_keywords": ["30"],
    },
    {
        "question": "What is the home-office stipend amount for new employees?",
        "expected_source": "company_handbook.md",
        "expected_keywords": ["300"],
    },
    {
        "question": "What encryption standard protects CloudSync Pro files at rest?",
        "expected_source": "product_faq.md",
        "expected_keywords": ["AES-256", "AES"],
    },
]


In [15]:
def evaluate(eval_set: list[dict]) -> dict:
    retrieval_hits = 0
    answer_hits = 0
    rows = []

    for item in eval_set:
        question = item["question"]
        retrieved = retriever.invoke(question)
        retrieved_sources = [d.metadata["source"] for d in retrieved]
        retrieval_correct = item["expected_source"] in retrieved_sources

        answer = rag_chain.invoke(question)
        answer_correct = any(kw.lower() in answer.lower() for kw in item["expected_keywords"])

        retrieval_hits += int(retrieval_correct)
        answer_hits += int(answer_correct)

        rows.append({
            "question": question,
            "expected_source": item["expected_source"],
            "retrieved_sources": retrieved_sources,
            "retrieval_correct": retrieval_correct,
            "answer": answer,
            "answer_correct": answer_correct,
        })

    n = len(eval_set)
    metrics = {
        "retrieval_accuracy": retrieval_hits / n,
        "answer_accuracy": answer_hits / n,
    }
    return metrics, rows


metrics, rows = evaluate(eval_set)

print("=== Per-question results ===\n")
for r in rows:
    print(f"Q: {r['question']}")
    print(f"  expected source : {r['expected_source']}")
    print(f"  retrieved       : {r['retrieved_sources']}  -> {'OK' if r['retrieval_correct'] else 'MISS'}")
    print(f"  answer          : {r['answer']!r}")
    print(f"  keyword match   : {'OK' if r['answer_correct'] else 'MISS'}")
    print()

print("=== Aggregate metrics ===")
print(f"Retrieval accuracy: {metrics['retrieval_accuracy']:.2%}")
print(f"Answer accuracy:    {metrics['answer_accuracy']:.2%}")


=== Per-question results ===

Q: How many vacation days do full-time employees accrue per year?
  expected source : company_handbook.md
  retrieved       : ['company_handbook.md', 'company_handbook.md', 'company_handbook.md']  -> OK
  answer          : '18 days'
  keyword match   : OK

Q: What is the price of the CloudSync Pro Business plan?
  expected source : product_faq.md
  retrieved       : ['product_faq.md', 'product_faq.md', 'product_faq.md']  -> OK
  answer          : '$15/user/month.'
  keyword match   : OK

Q: How long do new employees have to enroll in benefits?
  expected source : onboarding_guide.txt
  retrieved       : ['onboarding_guide.txt', 'onboarding_guide.txt', 'company_handbook.md']  -> OK
  answer          : '30 days from their start date.'
  keyword match   : OK

Q: What is the home-office stipend amount for new employees?
  expected source : company_handbook.md
  retrieved       : ['company_handbook.md', 'product_faq.md', 'company_handbook.md']  -> OK
  answer  

## Next steps / extensions

- Swap `google/flan-t5-base` for a larger instruction-tuned model for
  noticeably better answers (e.g. a 7B causal LM via `text-generation`).
- Replace the keyword-containment metric with **ROUGE-L** or an
  **LLM-as-judge** for nuanced answer scoring.
- Add more documents to `my_docs/` — the pipeline re-chunks and
  re-embeds automatically, no other code changes needed.
- Persist the FAISS index with `vectorstore.save_local(...)` /
  `FAISS.load_local(...)` to avoid re-embedding on every run.
- Tune `chunk_size`, `chunk_overlap`, and retriever `k` to balance
  context relevance vs. prompt length.
